# Compare Benchmark Charts

**Purpose:**
- Read Notebook 2 artifacts
- Generate comparison tables and charts without loading models
- Enforce schema validation

In [ ]:
import sys
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

_cwd = Path.cwd()
if _cwd.name != "notebooks":
    sys.path.insert(0, str(_cwd / "notebooks"))

import notebook_setup
ROOT = notebook_setup.setup_root()


## Configuration

In [ ]:
try:
    with open(ROOT / "results/charts/notebook_demo/latest_run.json", "r") as f:
        latest_run = json.load(f)
except FileNotFoundError:
    raise FileNotFoundError("latest_run.json not found. Please run Notebook 02 first.")

RUN_ID = latest_run["run_id"]
DATASET = latest_run["dataset"]
CSV_PATH = ROOT / latest_run["table_dir"] / f"{DATASET}_three_version.csv"
FIGURE_DIR = ROOT / latest_run["figure_dir"]

FIGURE_DIR.mkdir(parents=True, exist_ok=True)
print(f"Reading run {RUN_ID} for {DATASET}")


## Generate Charts

In [ ]:
from ccdf.demo.charting import generate_charts_for_dataset

if not CSV_PATH.exists():
    raise FileNotFoundError(f"Missing data: {CSV_PATH}")

print(f"Generating charts for {DATASET}...")

for filename, figure in generate_charts_for_dataset(str(CSV_PATH), DATASET):
    output_path = FIGURE_DIR / filename
    figure.savefig(output_path, dpi=160, bbox_inches="tight")
    
    display(figure)
    print(f"Saved: {output_path}")
    
    plt.show()
    plt.close(figure)

# Display table summary
df = pd.read_csv(CSV_PATH)
print(f"\n--- {DATASET.upper()} Data Summary ---")
summary_cols = ["condition", "t_e2e_ms", "generation_tok_s"]
if "numeric_match" in df.columns:
    summary_cols.append("numeric_match")
display(df[summary_cols].groupby("condition").mean().round(2))
print("\n")


## Interpretation
- Charts are saved to `results/charts/notebook_demo/figures/`.
- GSM8K displays numeric/strict exact match proxies.
- QMSum charts display runtime metrics, but **do not claim semantic correctness.** The target model's output quality on QMSum is a final limitation.